In [ ]:
# %pip install git+https://github.com/Mishne-Lab/pyRATS
# if not already installed on the system:
# %pip install pandas imageio seaborn matplotlib

from pyRATS import rats
from examples import vis

# Load data

In [ ]:
import os
import pickle

def path_exists(path):
    return os.path.exists(path) or os.path.islink(path)

def read(fpath, verbose=True):
    if not path_exists(fpath):
        if verbose:
            print(fpath, 'does not exist.')
        return None
    with open(fpath, "rb") as f:
        data = pickle.load(f)
    if verbose:
        print('Read data from', fpath, flush=True)
    return data


In [ ]:
data_fname = '../data/data.dat'             # PATH TO DATA
X, labels, metadata = read(data_fname)
cond_num = metadata['cond_num']

# Generate embeddings

In [ ]:
def f(x):
    z = x.copy()
    mask = x < 6
    z[mask] -= 2
    z[~mask] -= 1
    return z


model = rats.RATS(
    n_components = 2,
    n_neighbors = 45,
    cost_function = 'alignment',
    min_cluster_size=3,
    patience=20,
    n_iter=20,
    tear=True,
    postprocess=True,
    root_view='center',
    tree='spt',
    verbose=False
)

y, _ = model.fit_transform(X, cond_num)
tear_color_eig_inds = [1]
color_of_pts_on_tear = model.compute_color_of_pts_on_tear(
    y,
    tear_color_eig_inds=tear_color_eig_inds, 
)
mask_pos = cond_num > 0
mask_neg = ~mask_pos
vis.Visualize().global_embedding(
    y[mask_pos], f(cond_num[mask_pos]),
    color_of_pts_on_tear=color_of_pts_on_tear[mask_pos][:,tear_color_eig_inds] if color_of_pts_on_tear is not None else None,
    cmap0='bwr',
    cmap1='gist_ncar',
    title='color='+str([1]),
    figsize=(5, 5)
)

vis.Visualize().global_embedding(
    y[mask_neg], f(-cond_num[mask_neg]),
    color_of_pts_on_tear=color_of_pts_on_tear[mask_neg][:,tear_color_eig_inds] if color_of_pts_on_tear is not None else None,
    cmap0='bwr',
    cmap1='gist_ncar',
    title='color='+str([1]),
    figsize=(5, 5)
)
